## **Baseline Model Evaluation**

This notebook evaluates the winning solution from the [AICrowd Airborne Object Tracking Challenge](https://www.aicrowd.com/challenges/airborne-object-tracking-challenge) (2021) as the project baseline. The solution, developed by Dmytro Poplavskiy, uses a segmentation-based detection and tracking pipeline (SegTracker) with an ensemble of four deep learning models: HRNet32, HRNet48, GerNet-M with BiFPN, and DLA60.

Inference is performed across the entire custom hard dataset subset [outlined below](#dataset), with results presented both as an overall set of benchmark statistics and as performance metrics for each data subset.

**Evaluation Metrics:**
- **AFDR:** Airborne Flight Detection Rate. The primary detection metric.
- **FPPI:** False Positives Per Image. Must be ≤ 0.0005 to be ranked.
- **HFAR:** Horizontal False Alarm Rate. Must be ≤ 0.5 to be ranked.
- **EDR:** Encounter Detection Rate. Tracking-based metric across all horizon scenarios.

> ⚠️ ***Note:*** *Inference requires a CUDA-capable GPU. The `map_location` fix in `seg_tracker.py` allows CPU loading of weights for inspection, but full inference at acceptable speed requires GPU. Estimated runtime on GPU: ~2–3 FPS per flight sequence.*

In [ ]:
# Import Required Libraries.
import os
import sys
import json
import random
from glob import glob

import pandas as pd

# Because Life, Universe & Everything!
random.seed(42)

# Set Working Directory to Project Root.
os.chdir("..") 
PROJECT_ROOT = os.getcwd()

# Add Project Root to Path.
sys.path.insert(0, PROJECT_ROOT)

# Add Core Metric Folders to System Path.
sys.path.insert(0, os.path.join(PROJECT_ROOT, "core"))

# Import Local Project Funtions.
from logging_module import LogLevel, log
from src.baseline.seg_test import SegPredictor
from src.data.download_data import main as downloadData
from notebooks.notebook_helpers import get_next_run_dir, load_combined_gt, resolve_local_flights

# Confirm Project Root.
log(LogLevel.INFO, "baseline:imports", "Project Root Set: %s", PROJECT_ROOT)


---

<br/>

## **Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

**Isolation Criteria - encounters were selected for:**
- **Small Targets:** Airborne objects occupying only a few pixels per frame.
- **Multiple Targets:** Scenes containing more than one airborne object simultaneously.
- **Poor Visibility:** Flights affected by sun glare, adverse lighting, or challenging camera angles.

Together these conditions reflect the edge cases where sense-and-avoid systems are most likely to fail, and where reliable early detection is most critical for safe autonomous flight.

### **Airborne Object Tracking (AOT) - Hard Subset Summary**

| Metric                   | Value   | Description                                                             |
|--------------------------|--------:|-------------------------------------------------------------------------|
| **Total Flights**        | **1618**| **Total number of flight sequences in the dataset**                     |
| Small Object Flights     | 662     | Flights containing at least one object with bounding box area ≤ 36.0px².|
| Below Horizon Flights    | 1053    | Flights containing at least one object at or below the horizon.         |
| Multi-Object Flights     | 627     | Flights containing more than one airborne object in at least one frame. |
| No Object Frames         | 354     | Flights containing at least one frame with no airborne object present.  |
| *Avg. Sequence Length*   | *17.77s*| *Average duration of each flight sequence.*                             |

In [ ]:
# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "configs/datasets/hard_dataset_flights.txt"
DATASET_PARTS = ["part1", "part2", "part3"]

# Set Results Variables (Auto-Incrementing Run Directory).
RESULTS_DIR = "outputs/results/baseline/"
run_name = get_next_run_dir(RESULTS_DIR)

# Evaluation Filtering.
MIN_SCORE = 0.0
MIN_TRACK_LEN = 0

# Evaluation Output Folders.
EVAL_GT_DIR = os.path.join(RESULTS_DIR, run_name, "gt")
EVAL_RESULT_DIR  = os.path.join(RESULTS_DIR, run_name, "result")
EVAL_SUMMARY_DIR = os.path.join(RESULTS_DIR, run_name, "summaries")

# Print Summary Statement.
log(LogLevel.INFO, "baseline:config", "Configuration Loaded.")
print(f"  Dataset Path  : {DATASET_PATH}")
print(f"  Dataset Parts : {DATASET_PARTS}")
print(f"  Run Name      : {run_name}")
print(f"  Results Path  : {RESULTS_DIR}")
print(f"  Min Score     : {MIN_SCORE}")
print(f"  Min Track Len : {MIN_TRACK_LEN}")

In [ ]:
# Download the Selected Flight Sequences.
downloadData(DATASET_PATH)


---

<br/>

## **Data Preparation and Preprocessing**

Before inference can begin, the dataset must be resolved from the flight ID list down to what is actually available on disk, and the ground truth must be filtered to match. This cell performs two steps:

**1. Flight Resolution:** Parses the `.txt` flight ID file produced by `data_builder.py`, cross-references each entry against the local `data/<part>/Images/` directories, and builds a lookup of every flight that is both listed and downloaded. Flights in the list that have not been downloaded are logged and skipped.

**2. Ground Truth Filtering:** Loads the `groundtruth.json` file from each part's `ImageSets/` folder and merges them into a single structure, retaining only the entries that correspond to locally available flights. Entries for flights not on disk are silently skipped - this is expected, as the ground truth files reference the full AOT dataset rather than just the downloaded subset.

In [ ]:
# Find all Available Sequences from the Provided Dataset.
local_flights = resolve_local_flights(DATASET_PATH, PROJECT_ROOT)

# Add Spacing in Console Output for Readability.
print()

# Log How Many Flights were Found & are Ready for Inference.
log(LogLevel.INFO, "baseline:preprocessing", "Flights Ready for Inference: %d", len(local_flights))

# Preview the First 5 Flights to Confirm Dataset is Correct.
for name, info in list(local_flights.items())[:5]:
    # Count No. Images in the Flight Directory.
    n_imgs = len(list(info["images_dir"].glob("*.png")))

    # Print Flight Part, Name, & No. Images.
    print(f"  [{info['part']}]  {name}  ({n_imgs} images)")

# If there are More than 5 Flights, Log How Many Additional Ones Exist.
if len(local_flights) > 5:
    log(LogLevel.DEBUG, "baseline:preprocessing", " ... and %d more.", len(local_flights) - 5)

# Load & Filter Ground Truth to Include Local Flights Only. 
combined_gt = load_combined_gt(DATASET_PARTS, local_flights, PROJECT_ROOT)
log(LogLevel.INFO, "baseline:preprocessing", "Flights in Filtered Ground Truth: %d", len(combined_gt['samples']))

# Save the Filtered Ground Truth to the Evaluation Folder.
os.makedirs(EVAL_GT_DIR, exist_ok=True)
gt_out_path = os.path.join(EVAL_GT_DIR, "groundtruth.json")

# Copy the Ground Truth Values to the .json.
with open(gt_out_path, "w") as f:
    json.dump(combined_gt, f)

# Print Success Message.
log(LogLevel.INFO, "baseline:preprocessing", "Ground Truth Saved: %s", gt_out_path)


---

<br/>

## **Evaluate Baseline Model**

### **Run Baseline Inference**

**1.** Prepare evaluation directory and configure environment variables.

In [ ]:
# Create a Directory where all Evaluation Flight Data will be Stored.
EVAL_DATA_DIR = os.path.join(RESULTS_DIR, run_name, "data")
os.makedirs(EVAL_DATA_DIR, exist_ok=True)

# Link Each Local Flight's Image Folder into the Evaluation Directory.
for folder_name, info in local_flights.items():
    link_path = os.path.join(EVAL_DATA_DIR, folder_name)
    if not os.path.exists(link_path):
        os.symlink(info["images_dir"], link_path)

# Set Environment Variables that AirbornePredictor and SegPredictor Read.
os.environ["TEST_DATASET_PATH"] = EVAL_DATA_DIR
os.environ["INFERENCE_OUTPUT_PATH"] = RESULTS_DIR
os.environ["DATASET_ENV"] = run_name
os.environ["MIN_TRACK_LEN"] = str(MIN_TRACK_LEN)
os.environ["MIN_SCORE"] = str(MIN_SCORE)

# Limit Inference to Only the Specified Flights.
os.environ["PARTIAL_RUN_FLIGHTS"] = ",".join(local_flights.keys())

# Print Summary Statement.
log(LogLevel.INFO, "baseline:inference", "Environment Configured.")
print(f"  Test Data Path     : {EVAL_DATA_DIR}")
print(f"  Flights to Process : {len(local_flights)}")
print(f"  MIN_SCORE          : {MIN_SCORE}")
print(f"  MIN_TRACK_LEN      : {MIN_TRACK_LEN}")

**2.** Run segmentation predictor on all evaluation flights.

> ⚠️ ***This cells requires a CUDA GPU and will take several hours to complete.*** *Skip this cell if inference has already been run and results are present in `RESULTS_DIRS`.*

In [ ]:
# Initialise Predictor (Reads Env Variables).
predictor = SegPredictor()

# Run Inference Over all Flights.
predictor.run()

# Print Summary Statement.
log(LogLevel.INFO, "baseline:inference", "Inference Complete.")
print(f"  Raw Results Saved to: {os.path.join(RESULTS_DIR, run_name)}")

**3.** Convert bounding boxes from corner format to centre format.

In [ ]:
# Set Results Directories. 
RAW_RESULTS_PATH = os.path.join(RESULTS_DIR, run_name, "result.json")
METRICS_RESULTS_PATH = os.path.join(EVAL_RESULT_DIR, "result.json")

# Load Raw Inference Results.
with open(RAW_RESULTS_PATH) as f:
    flight_results = json.load(f)

# Convert Each Detection from Corner Format to Centre Format.
for entry in flight_results:
    for det in entry.get("detections", []):
        # Extract Corner Coordinates.
        x1, y1, x2, y2 = det["x"], det["y"], det["w"], det["h"]

        # Compute Width & Height.
        w = x2 - x1
        h = y2 - y1

        # Convert to Center Coordinates.
        det["x"] = x1 + w / 2
        det["y"] = y1 + h / 2
        det["w"] = w
        det["h"] = h

# Ensure Output Directory Exists.
os.makedirs(EVAL_RESULT_DIR, exist_ok=True)

# Save Converted Results in Metrics-Compatible Format.
with open(METRICS_RESULTS_PATH, "w") as f:
    json.dump(flight_results, f)

# Print Summary Statement.
log(LogLevel.INFO, "baseline:inference", "Results Converted and Saved.")
print(f"  Metrics-Ready Results Stored at : {METRICS_RESULTS_PATH}")

### **Run Metrics Evaluation**

Run the official AICrowd evaluation metrics against the ground truth for each dataset part, saving per-part summaries to `SUMMARIES_DIR`.

In [ ]:
# Locate the Competition Metrics Script.
metrics_script = os.path.join(PROJECT_ROOT, "core/metrics/run_airborne_metrics.py")

# Call the Competition Metrics Script.
!python3 {metrics_script} \
    --dataset-folder   {EVAL_GT_DIR} \
    --results-folder   {EVAL_RESULT_DIR} \
    --summaries-folder {EVAL_SUMMARY_DIR}

# Print Message on Completion. 
log(LogLevel.INFO, "baseline:evaluation", "Metrics Complete.")

# -- END --

### **Baseline Benchmark Summary**

Display the aggregated statistics across all three parts of the dataset in order to provide an overall baseline performance for the full hard dataset. Additionally, statistics for each individual subset of the hard dataset are also displayed for more detailed analysis.

In [ ]:
# Check that the Summary Files Exist. 
if not all_summaries:
    log(LogLevel.ERROR, "baseline:summary", "No Summary Files Found! Run Steps 1, 2 and 3 First.")
else:
    # # Aggregate Metrics Across All Parts.
    total_detected   = df_parts["Detected Aircraft"].sum()
    total_aircraft   = df_parts["Total Aircraft"].sum()
    mean_fppi        = df_parts["FPPI"].mean()
    mean_hfar        = df_parts["HFAR"].mean()
    overall_afdr     = total_detected / total_aircraft if total_aircraft > 0 else 0
    mean_edr_all     = df_parts["EDR (All)"].mean()
    mean_edr_above   = df_parts["EDR (Above)"].mean()
    mean_edr_below   = df_parts["EDR (Below)"].mean()

    # Rank Metrics Against Thresholds and Display Results.
    if mean_fppi <= 0.0005:
        ranked_fppi = log(LogLevel.INFO, "baseline:summary", "FPPI Within Acceptable Limit (≤ 0.0005).")
    else:
        ranked_fppi = log(LogLevel.WARNING, "baseline:summary", f"FPPI Exceeds Limit! Actual: {mean_fppi:.6f} (≤ 0.0005).")
    if mean_hfar <= 0.5:
        ranked_hfar = log(LogLevel.INFO, "baseline:summary", "HFAR Within Acceptable Limit (≤ 0.5).")
    else:
        ranked_hfar = log(LogLevel.WARNING, "baseline:summary", f"HFAR Exceeds Limit! Actual: {mean_hfar:.4f} (≤ 0.5).")

    # Build a Summary DataFrame for Display and CSV Export.
    summary_df = pd.DataFrame([
        {"Metric": "AFDR (Airborne Flight Detection Rate)",  "Value": f"{overall_afdr:.4f}",  "Status": ""},
        {"Metric": "FPPI (False Positives Per Image)",       "Value": f"{mean_fppi:.6f}",     "Status": ranked_fppi},
        {"Metric": "HFAR (Horizontal False Alarm Rate)",     "Value": f"{mean_hfar:.4f}",     "Status": ranked_hfar},
        {"Metric": "EDR - All Scenarios",                    "Value": f"{mean_edr_all:.4f}",  "Status": ""},
        {"Metric": "EDR - Above Horizon",                    "Value": f"{mean_edr_above:.4f}","Status": ""},
        {"Metric": "EDR - Below Horizon",                    "Value": f"{mean_edr_below:.4f}","Status": ""},
        {"Metric": "Total Aircraft Detected",                "Value": f"{int(total_detected)}","Status": ""},
        {"Metric": "Total Aircraft in Dataset",              "Value": f"{int(total_aircraft)}","Status": ""},
    ])

    # Print Summary to Console.
    print("\n" + "="*60)
    print(" BASELINE MODEL - FULL HARD DATASET BENCHMARK SUMMARY")
    print(" Model: Dmytro Poplavskiy Winning Solution (AICrowd 2021)")
    print("="*60)
    display(summary_df.set_index("Metric"))

    # Save Aggregated Summary to CSV.
    output_path = os.path.join(SUMMARIES_DIR, "baseline_benchmark_summary.csv")
    summary_df.to_csv(output_path, index=False)
    print()
    log(LogLevel.INFO, "baseline:summary", "Summary saved to: %s", output_path)


---

<br/>

## **Hard Dataset Results**

The table below reports baseline performance broken down by scenario category across the custom
hard dataset (1,618 flights). Flights were selected from the full AOT dataset using configurable
filter criteria to isolate the most challenging sequences, providing a more targeted benchmark
than the full competition dataset.

| Scenario                     | Flights | FPPI | AFDR | HFAR | EDR |
|------------------------------|---------|------|------|------|-----|
| **Overall (All Criteria)**   |**1,618**| —    | —    | —    | —   |
| Small Object (bbox ≤ 36 px²) | 662     | —    | —    | —    | —   |
| Below Horizon                | 1,053   | —    | —    | —    | —   |
| Multi-Object                 | 627     | —    | —    | —    | —   |
| No Visible Object            | 354     | —    | —    | —    | —   |

*Results evaluated at a minimum detection confidence of 0.985 and a minimum track length of 30 frames (≈ 3 seconds at 10 fps).*

---